# Bắt đầu với dữ liệu màu mẫu (Leanbot + APDS-9960)

Notebook này chỉ **đọc** dữ liệu trong `samples/leanbot-basic/data/`, nên chạy được ngay cả khi nằm trong thư mục chỉ đọc
`samples/`. Chọn **Run → Run All Cells**. Muốn sửa và lưu, hãy sao chép thư mục `leanbot-basic` sang thư mục của bạn trước
(xem `samples/README.md`).

In [ ]:
import glob, os
import pandas as pd

# Tìm thư mục data/ dù notebook nằm trong samples/leanbot-basic/ hay trong bản sao của bạn
DATA = "data" if os.path.isdir("data") else "/home/jovyan/work/samples/leanbot-basic/data"
files = sorted(glob.glob(os.path.join(DATA, "*.json")))
print("Đọc", len(files), "tệp:", [os.path.basename(f) for f in files])
df = pd.concat([pd.read_json(f) for f in files], ignore_index=True)
print(len(df), "mẫu")
df.head()

In [ ]:
# Số mẫu theo màu (bộ dữ liệu cân bằng: mỗi màu như nhau)
df["objColor"].value_counts()

In [ ]:
import matplotlib.pyplot as plt
df.groupby("objColor")[["objRed", "objGreen", "objBlue"]].mean().plot.bar(color=["red", "green", "blue"], figsize=(8, 4))
plt.title("Giá trị RGB trung bình theo màu vật thể"); plt.ylabel("số đo cảm biến"); plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

df = df[(df[["objRed", "objGreen", "objBlue"]] >= 0).all(axis=1)]          # bỏ mẫu đo lỗi (giá trị âm)
X, y = df[["objRed", "objGreen", "objBlue"]], df["objColor"]
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, stratify=y, random_state=13)
model = DecisionTreeClassifier(max_depth=5).fit(X_train.values, y_train)
print("Độ chính xác trên tập kiểm tra:", round(model.score(X_test.values, y_test), 3))
print(export_text(model, feature_names=["objRed", "objGreen", "objBlue"]))

Bước tiếp theo: chạy chương trình đầy đủ `code/dtree.py` (vẽ cây, ma trận nhầm lẫn, lưu `dtree.jpg`) trong **bản sao** của
bạn: xem `samples/leanbot-basic/README.md`.